In [ ]:
!pip install openpyxl

In [ ]:
from google.colab import drive
import pandas as pd
import openpyxl
from google.colab import drive
import os
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
#mount drive
drive.mount('/content/drive')

## (To be) Downloaded images

In [ ]:
# overall list of to be downloaded files
alltobedownloaded_path = '/content/drive/MyDrive/Thesis/images_dataset_data_description/AllPoliticians_captions.xlsx'
alltobedownloaded = pd.read_excel(alltobedownloaded_path)

In [ ]:
# takes 13 min
# get downloaded images
folder_path = '/content/drive/MyDrive/Thesis/images_dataset/origin'

jpg_paths = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.lower().endswith(".jpg"):
            file_path = os.path.join(root, file)
            jpg_paths.append(file_path)

df_downloaded_images = pd.DataFrame(jpg_paths, columns=["image_path"])

## to be generated images

In [ ]:
alltobegenerated = alltobedownloaded.copy()

In [ ]:
# get generated images
folder_path = '/content/drive/MyDrive/Thesis/images_generated'

jpg_paths = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.lower().endswith(".jpg"):
            file_path = os.path.join(root, file)
            jpg_paths.append(file_path)

df_generated_images = pd.DataFrame(jpg_paths, columns=['image_path'])
df_generated_images.to_excel('/content/drive/MyDrive/Thesis/generated_images.xlsx')

In [ ]:
# strip path from washington_post_images_0012_417_2_google.jpg to ./washington_post/images/0012/417.jpg
def strip_path(path):
    filename = path.split('/')[-1]
    parts = filename.replace('.jpg', '').split('_')
    prefix = '_'.join(parts[:-4])
    return f"./{prefix}/images/{parts[-3]}/{parts[-2]}.jpg"
df_generated_images['image_path_stripped'] = df_generated_images['image_path'].apply(strip_path)

#join stripped_path to df
generated_images = alltobedownloaded.merge(
    df_generated_images,
    how='inner',
    left_on='image_path',
    right_on='image_path_stripped'
)
generated_images.to_excel('/content/drive/MyDrive/Thesis/generated_images.xlsx')

In [ ]:
#get only paths that have 5 generated images
grouped_counts = df_generated_images.groupby('image_path_stripped').size().reset_index(name='count')


In [ ]:
# create tobegenerated df that still do not have
tobegenerated = alltobegenerated[~alltobegenerated['image_path'].isin(grouped_counts['image_path_stripped'])]
tobegenerated = tobegenerated.sample(frac=1).reset_index(drop=True)
tobegenerated.to_excel('/content/drive/MyDrive/Thesis/tobegenerated.xlsx')


## Google generated images


In [ ]:
Get google top 5 images
folder_path = '/content/drive/MyDrive/Thesis/images_google'
jpg_paths = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.lower().endswith(".jpg"):
            file_path = os.path.join(root, file)
            jpg_paths.append(file_path)

df_generated_google_images = pd.DataFrame(jpg_paths, columns=['image_path'])

In [ ]:
# strip path from washington_post_images_0012_417_2_google.jpg to ./washington_post/images/0012/417.jpg
def strip_path_google(path):
    filename = os.path.basename(path).replace('.jpg', '')
    parts = filename.split('_')
    try:
        images_idx = parts.index('images')
    except ValueError:
        return None

    source = '_'.join(parts[:images_idx])

    if len(parts) >= images_idx + 4:
        folder = parts[images_idx + 1]
        image_name = parts[images_idx + 2]
        return f"./{source}/images/{folder}/{image_name}.jpg"
    else:
        return None

df_generated_google_images['image_path_stripped'] = df_generated_google_images['image_path'].apply(strip_path_google)

#get images that have 5 or more google top 5 images
grouped_counts_google = df_generated_google_images.groupby('image_path_stripped').size().reset_index(name='count')
grouped_counts_google[grouped_counts_google['count']>=5].to_csv('/content/drive/MyDrive/Thesis/generated_google_images.csv')
